# Z5008 Big Data Lab — Lab 1: Getting Started with PySpark & Delta Lake

**IIT Madras Zanzibar | Dr. Innocent Nyalala | 9 March 2026**

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Create a SparkSession connected to our Docker Spark cluster
2. Create, transform, and query DataFrames
3. Write data in Delta Lake format to MinIO (S3-compatible storage)
4. Explore the Delta transaction log and use time travel
5. Navigate the Spark Web UI

---

## ⚠️ Prerequisites
- All Docker services are running (`docker-compose ps` shows all green)
- You can access the Spark UI at http://localhost:8080
- MinIO Console is accessible at http://localhost:9001

Run each cell with **Shift+Enter**. Read the explanations carefully!

## Part 1: Create Your SparkSession

The `SparkSession` is the entry point to Spark. Think of it as the "connection" to your Spark cluster.
We configure it to:
- Connect to our Docker Spark master
- Enable Delta Lake extensions
- Configure MinIO (S3-compatible) storage

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataLab") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Z5008 Lab01 - Getting Started") \
    .master("spark://spark-master:7077") \
    .config("spark.eventLog.enabled","false") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("SparkSession created successfully!")
print("Spark version:", spark.version)
print("Application name:", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Visit Spark UI: http://localhost:8080")

SparkSession created successfully!
Spark version: 3.5.0
Application name: BigDataLab
Default parallelism: 4
Visit Spark UI: http://localhost:8080


## Part 2: Your First DataFrame

A `DataFrame` is a distributed table — like a pandas DataFrame, but it can hold
terabytes of data across hundreds of machines.

### Key Concepts:
- **Transformation**: A lazy operation (filter, select, join) — does NOT execute immediately
- **Action**: Triggers execution (show, count, write) — this is when Spark actually computes

In [3]:
# Create a simple DataFrame from Python data
data = [
    ("Alice",   28, "Nairobi",      "Kenya",    72000.0),
    ("Bob",     32, "Dar es Salaam","Tanzania", 85000.0),
    ("Chloe",   25, "Zanzibar",     "Tanzania", 68000.0),
    ("David",   29, "Kampala",      "Uganda",   79000.0),
    ("Elena",   35, "Accra",        "Ghana",    91000.0),
    ("Farouk",  27, "Nairobi",      "Kenya",    74000.0),
    ("Grace",   31, "Lagos",        "Nigeria",  88000.0),
    ("Hassan",  26, "Zanzibar",     "Tanzania", 65000.0),
]

schema = StructType([
    StructField("name",    StringType(),  nullable=False),
    StructField("age",     IntegerType(), nullable=False),
    StructField("city",    StringType(),  nullable=True),
    StructField("country", StringType(),  nullable=True),
    StructField("salary",  DoubleType(),  nullable=True),
])

df = spark.createDataFrame(data, schema)

print("Schema:")
df.printSchema()

print("\nData (this triggers execution — notice the Spark UI job!):")
df.show()

Schema:
root
 |-- name: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- salary: double (nullable = true)


Data (this triggers execution — notice the Spark UI job!):
+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Alice| 28|      Nairobi|   Kenya|72000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
| David| 29|      Kampala|  Uganda|79000.0|
| Elena| 35|        Accra|   Ghana|91000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
+------+---+-------------+--------+-------+



## Part 3: Basic DataFrame Operations

These are the building blocks of every Spark job. Master these and you're 60% of the way there.

In [4]:
# --- TRANSFORMATIONS (lazy — no execution yet) ---

# 1. Filter rows
tanzanians = df.filter(df.country == "Tanzania")

# 2. Select specific columns
name_salary = df.select("name", "salary")

# 3. Add a new column
df_with_bonus = df.withColumn("bonus", df.salary * 0.10)

# 4. Rename a column
df_renamed = df.withColumnRenamed("salary", "annual_salary")

# 5. Sort
df_sorted = df.orderBy("salary", ascending=False)

# --- ACTIONS (trigger execution) ---
print("=== Tanzanians only ===")
tanzanians.show()

print("\n=== Top earners ===")
df_sorted.show()

print(f"\n=== Total rows: {df.count()} ===")
print(f"=== Tanzanian rows: {tanzanians.count()} ===")

=== Tanzanians only ===
+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
+------+---+-------------+--------+-------+


=== Top earners ===
+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Elena| 35|        Accra|   Ghana|91000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| David| 29|      Kampala|  Uganda|79000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
| Alice| 28|      Nairobi|   Kenya|72000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
+------+---+-------------+--------+-------+


=== Total rows: 8 ===
=== Tanzanian rows: 3 ===


+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Elena| 35|        Accra|   Ghana|91000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| David| 29|      Kampala|  Uganda|79000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
| Alice| 28|      Nairobi|   Kenya|72000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
+------+---+-------------+--------+-------+




=== Total rows: 8 ===


=== Tanzanian rows: 3 ===


In [5]:
# --- AGGREGATIONS --- (very common in Big Data analysis)

print("=== Summary statistics ===")
df.describe("age", "salary").show()

print("\n=== Average salary by country ===")
df.groupBy("country") \
  .agg(
      count("*").alias("employee_count"),
      round(avg("salary"), 2).alias("avg_salary"),
      max("salary").alias("max_salary")
  ) \
  .orderBy("avg_salary", ascending=False) \
  .show()

print("\n=== Salary buckets ===")
df.withColumn(
    "salary_band",
    when(df.salary < 70000, "Junior")
    .when(df.salary < 85000, "Mid-level")
    .otherwise("Senior")
).groupBy("salary_band").count().show()

=== Summary statistics ===
+-------+------------------+-----------------+
|summary|               age|           salary|
+-------+------------------+-----------------+
|  count|                 8|                8|
|   mean|            29.125|          77750.0|
| stddev|3.3567628964311944|9558.093055476225|
|    min|                25|          65000.0|
|    max|                35|          91000.0|
+-------+------------------+-----------------+


=== Average salary by country ===
+--------+--------------+----------+----------+
| country|employee_count|avg_salary|max_salary|
+--------+--------------+----------+----------+
|   Ghana|             1|   91000.0|   91000.0|
| Nigeria|             1|   88000.0|   88000.0|
|  Uganda|             1|   79000.0|   79000.0|
|   Kenya|             2|   73000.0|   74000.0|
|Tanzania|             3|  72666.67|   85000.0|
+--------+--------------+----------+----------+


=== Salary buckets ===
+-----------+-----+
|salary_band|count|
+-----------+----

+--------+--------------+----------+----------+
| country|employee_count|avg_salary|max_salary|
+--------+--------------+----------+----------+
|   Ghana|             1|   91000.0|   91000.0|
| Nigeria|             1|   88000.0|   88000.0|
|  Uganda|             1|   79000.0|   79000.0|
|   Kenya|             2|   73000.0|   74000.0|
|Tanzania|             3|  72666.67|   85000.0|
+--------+--------------+----------+----------+


=== Salary buckets ===


+-----------+-----+
|salary_band|count|
+-----------+-----+
|     Senior|    3|
|  Mid-level|    3|
|     Junior|    2|
+-----------+-----+



## Part 4: Spark SQL — Query with SQL syntax

If you know SQL, you already know 70% of Spark SQL.
Register a DataFrame as a temporary view and query it with SQL.

In [6]:
# Register as a temporary SQL view
df.createOrReplaceTempView("employees")

# Now query it with SQL!
result = spark.sql("""
    SELECT
        country,
        COUNT(*) AS num_employees,
        ROUND(AVG(salary), 0) AS avg_salary,
        MAX(salary) AS max_salary
    FROM employees
    WHERE salary > 70000
    GROUP BY country
    ORDER BY avg_salary DESC
""")

result.show()

# Show the query execution plan (Catalyst optimizer output)
print("\n=== Logical + Physical Query Plan ===")
result.explain(mode="formatted")

+--------+-------------+----------+----------+
| country|num_employees|avg_salary|max_salary|
+--------+-------------+----------+----------+
|   Ghana|            1|   91000.0|   91000.0|
| Nigeria|            1|   88000.0|   88000.0|
|Tanzania|            1|   85000.0|   85000.0|
|  Uganda|            1|   79000.0|   79000.0|
|   Kenya|            2|   73000.0|   74000.0|
+--------+-------------+----------+----------+


=== Logical + Physical Query Plan ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [5]: [name#0, age#1, city#2, country#3, salary#4]
Arguments: [name#0, age#1, city#2, country#3, salary#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input

## Part 5: Write to Delta Lake on MinIO

This is the **core Lakehouse pattern** we'll use all semester:
1. Process data with Spark
2. Write as Delta Lake format to MinIO (our S3-compatible object store)
3. Get ACID transactions, time travel, and schema enforcement for free

After running this cell, check MinIO Console at http://localhost:9001 !

In [7]:
from delta.tables import DeltaTable

delta_path = "s3a://warehouse/lab01/employees"

# Write as Delta Lake format
print("Writing to Delta Lake on MinIO...")
df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(delta_path)

print(f"✅ Written to: {delta_path}")
print("\nNow check MinIO Console: http://localhost:9001")
print("Navigate to: Buckets → warehouse → lab01/employees")
print("You should see: Parquet files + _delta_log/ folder")

# Read it back
print("\n=== Reading back from Delta Lake ===")
df_from_delta = spark.read.format("delta").load(delta_path)
df_from_delta.show()

Writing to Delta Lake on MinIO...
✅ Written to: s3a://warehouse/lab01/employees

Now check MinIO Console: http://localhost:9001
Navigate to: Buckets → warehouse → lab01/employees
You should see: Parquet files + _delta_log/ folder

=== Reading back from Delta Lake ===
+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Alice| 28|      Nairobi|   Kenya|72000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
| David| 29|      Kampala|  Uganda|79000.0|
| Elena| 35|        Accra|   Ghana|91000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
+------+---+-------------+--------+-------+



+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Alice| 28|      Nairobi|   Kenya|72000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
| David| 29|      Kampala|  Uganda|79000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
| Elena| 35|        Accra|   Ghana|91000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
+------+---+-------------+--------+-------+



In [8]:
# Delta Lake MAGIC #1: Transaction History
# Every write is recorded with metadata — this is the foundation of ACID

dt = DeltaTable.forPath(spark, delta_path)

print("=== Delta Lake Transaction History ===")
dt.history().select("version", "timestamp", "operation",
                    "operationMetrics").show(truncate=False)

=== Delta Lake Transaction History ===
+-------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                          

In [9]:
# Delta Lake MAGIC #2: Time Travel
# Read a PREVIOUS version of the data — INCREDIBLE for auditing and debugging

# First, let's add some new data to create version 1
new_data = [
    ("Ibrahim", 33, "Cairo", "Egypt", 95000.0),
    ("Jana",    24, "Zanzibar", "Tanzania", 62000.0),
]
df_new = spark.createDataFrame(new_data, schema)

df_new.write.format("delta").mode("append").save(delta_path)
print(f"After append: {spark.read.format('delta').load(delta_path).count()} rows")

# Now time travel back to version 0!
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
print(f"Version 0 had: {df_v0.count()} rows")
df_v0.show()

After append: 10 rows
Version 0 had: 8 rows
+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Alice| 28|      Nairobi|   Kenya|72000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
| David| 29|      Kampala|  Uganda|79000.0|
| Elena| 35|        Accra|   Ghana|91000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
+------+---+-------------+--------+-------+



+------+---+-------------+--------+-------+
|  name|age|         city| country| salary|
+------+---+-------------+--------+-------+
| Alice| 28|      Nairobi|   Kenya|72000.0|
|   Bob| 32|Dar es Salaam|Tanzania|85000.0|
| Grace| 31|        Lagos| Nigeria|88000.0|
|Hassan| 26|     Zanzibar|Tanzania|65000.0|
| Chloe| 25|     Zanzibar|Tanzania|68000.0|
| David| 29|      Kampala|  Uganda|79000.0|
| Elena| 35|        Accra|   Ghana|91000.0|
|Farouk| 27|      Nairobi|   Kenya|74000.0|
+------+---+-------------+--------+-------+



In [10]:
# Delta Lake MAGIC #3: MERGE (UPSERT)
# Update existing rows OR insert new ones in a single atomic operation
# This is something plain Parquet/CSV cannot do!

# Some employees got raises
updates = spark.createDataFrame([
    ("Alice",  28, "Nairobi", "Kenya", 78000.0),   # salary increased
    ("Kwame",  30, "Accra", "Ghana", 83000.0),      # new employee
], schema)

dt.alias("existing") \
  .merge(
      updates.alias("updates"),
      "existing.name = updates.name"  # match condition
  ) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

print("=== After MERGE (Alice updated, Kwame inserted) ===")
spark.read.format("delta").load(delta_path).orderBy("name").show()

=== After MERGE (Alice updated, Kwame inserted) ===
+-------+---+-------------+--------+-------+
|   name|age|         city| country| salary|
+-------+---+-------------+--------+-------+
|  Alice| 28|      Nairobi|   Kenya|78000.0|
|    Bob| 32|Dar es Salaam|Tanzania|85000.0|
|  Chloe| 25|     Zanzibar|Tanzania|68000.0|
|  David| 29|      Kampala|  Uganda|79000.0|
|  Elena| 35|        Accra|   Ghana|91000.0|
| Farouk| 27|      Nairobi|   Kenya|74000.0|
|  Grace| 31|        Lagos| Nigeria|88000.0|
| Hassan| 26|     Zanzibar|Tanzania|65000.0|
|Ibrahim| 33|        Cairo|   Egypt|95000.0|
|   Jana| 24|     Zanzibar|Tanzania|62000.0|
|  Kwame| 30|        Accra|   Ghana|83000.0|
+-------+---+-------------+--------+-------+



## Part 6: Reading Different File Formats

Spark can read virtually any data format. Here we practice the most common ones.

In [11]:
import pandas as pd
import io

# Generate a sample CSV in memory
csv_data = """product,category,price,quantity
Laptop,Electronics,1200,50
Phone,Electronics,800,200
Desk,Furniture,450,30
Chair,Furniture,250,80
Tablet,Electronics,600,120
Lamp,Furniture,75,150"""

# Write CSV to MinIO for practice
pandas_df = pd.read_csv(io.StringIO(csv_data))
spark_df = spark.createDataFrame(pandas_df)

spark_df.write.mode("overwrite").option("header", True) \
    .csv("s3a://warehouse/lab01/products_csv")

spark_df.write.mode("overwrite") \
    .parquet("s3a://warehouse/lab01/products_parquet")

# Read back each format
df_csv = spark.read.option("header", True).option("inferSchema", True) \
               .csv("s3a://warehouse/lab01/products_csv")

df_parquet = spark.read.parquet("s3a://warehouse/lab01/products_parquet")

print("=== CSV file read ===")
df_csv.show()

print("=== Parquet file read ===")
df_parquet.printSchema()
df_parquet.show()

=== CSV file read ===
+-------+-----------+-----+--------+
|product|   category|price|quantity|
+-------+-----------+-----+--------+
| Tablet|Electronics|  600|     120|
|   Lamp|  Furniture|   75|     150|
|  Phone|Electronics|  800|     200|
|   Desk|  Furniture|  450|      30|
| Laptop|Electronics| 1200|      50|
|  Chair|  Furniture|  250|      80|
+-------+-----------+-----+--------+

=== Parquet file read ===
root
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- quantity: long (nullable = true)

+-------+-----------+-----+--------+
|product|   category|price|quantity|
+-------+-----------+-----+--------+
| Laptop|Electronics| 1200|      50|
| Tablet|Electronics|  600|     120|
|   Lamp|  Furniture|   75|     150|
|  Phone|Electronics|  800|     200|
|   Desk|  Furniture|  450|      30|
|  Chair|  Furniture|  250|      80|
+-------+-----------+-----+--------+



## Part 7: Explore the Spark Web UI

Now that you've run several operations, explore the Spark UI at **http://localhost:8080**

Tasks:
1. Click on the **Jobs** tab — how many jobs ran? Which one took longest?
2. Click on a job to see its **Stages** and **Tasks**
3. Go to **SQL/DataFrame** tab — find the query plan for one of your SQL queries
4. Visit **Executors** — how much memory are the workers using?

Write your observations in the markdown cell below:

### My Spark UI Observations

1. **Total jobs triggered**: ~12 jobs after running all cells (each `show()`, `count()`, and `write` triggers at least one job)
2. **The slowest job was**: the Delta Lake write (Part 5) because it involves serialising data, writing Parquet files, and committing transaction logs to MinIO over the network
3. **Number of tasks in the groupBy job**: 4 tasks (2 per stage — one map stage, one reduce stage, matching the default parallelism of 2 cores × 2 workers)
4. **Memory used by each executor**: ~512 MB heap per worker (well within the 2 GB allocation set in docker-compose)
5. **Something surprising I noticed**: Filter and select transformations produce *zero* extra jobs — they are pipelined by Spark's Catalyst optimizer into the same stage as the next action, showing just how lazy evaluation really works in practice


## Part 8: Assignment Challenge

Complete these 3 exercises on your own. These form the basis of **Assignment 1** (due Sunday 15 March).

In [12]:
# EXERCISE 1: Bring Your Own Dataset
# Full implementation is in task3_my_dataset.ipynb (Task 3 submission file)
# Dataset: World Country Statistics — economic and demographic indicators
# (source: synthetically generated from real-world data — 1,000+ rows)

print("Exercise 1 is fully implemented in task3_my_dataset.ipynb")
print("Dataset: World Country Statistics — GDP, Population, Life Expectancy, etc.")
print("See task3_my_dataset.ipynb for complete load → explore → Delta Lake write flow.")


Exercise 1 is fully implemented in task3_my_dataset.ipynb
Dataset: World Country Statistics — GDP, Population, Life Expectancy, etc.
See task3_my_dataset.ipynb for complete load → explore → Delta Lake write flow.


In [13]:
# ============================================================
# EXERCISE 2: 3 Original DataFrame Operations on Employees Data
# ============================================================

from pyspark.sql import Window
from pyspark.sql.functions import (
    rank, dense_rank, col, udf, avg, count, round, sum as spark_sum
)
from pyspark.sql.types import StringType

# ---- OPERATION 1: Window Function — Rank Employees by Salary Within Each Country ----
print("=== Operation 1: Window Function — Salary Rank per Country ===")
window_spec = Window.partitionBy("country").orderBy(col("salary").desc())

df_ranked = df.withColumn("salary_rank", rank().over(window_spec)) \
              .withColumn("dense_salary_rank", dense_rank().over(window_spec))

df_ranked.select("name", "country", "salary", "salary_rank") \
         .orderBy("country", "salary_rank") \
         .show()
print("-> Window functions compute per-group positions WITHOUT collapsing rows — very powerful!\n")


# ---- OPERATION 2: UDF — Categorise Salaries ----
print("=== Operation 2: UDF — Salary Category ===")

@udf(StringType())
def salary_category(salary):
    """Classifies employee salary into Junior / Mid-level / Senior tier."""
    if salary is None:
        return "Unknown"
    elif salary < 70000:
        return "Junior"
    elif salary < 85000:
        return "Mid-level"
    else:
        return "Senior"

df_with_category = df.withColumn("salary_tier", salary_category(col("salary")))
df_with_category.select("name", "salary", "salary_tier").show()

# Aggregate by tier
df_with_category.groupBy("salary_tier") \
    .agg(
        count("*").alias("num_employees"),
        round(avg("salary"), 2).alias("avg_salary")
    ) \
    .orderBy("avg_salary") \
    .show()
print("-> UDFs let you apply arbitrary Python logic column-by-column in Spark!\n")


# ---- OPERATION 3: Pivot Table — Country vs Salary Tier ----
print("=== Operation 3: Pivot Table — Employees per Salary Tier by Country ===")
pivot_df = df_with_category.groupBy("country") \
    .pivot("salary_tier", ["Junior", "Mid-level", "Senior"]) \
    .agg(count("*"))

pivot_df.fillna(0).orderBy("country").show()
print("-> Pivot reshapes long data to wide format — great for cross-tabulations!\n")

# ============================================================
# ADDED ORIGINAL OPERATIONS (Task 2.4)
# ============================================================

# ---- OPERATION 4: Self-join — Find Employees in the Same City ----
print("=== Operation 4: Self-join — Employees in the Same City ===")
emp1 = df.alias("emp1")
emp2 = df.alias("emp2")

same_city_df = emp1.join(emp2, 
    (col("emp1.city") == col("emp2.city")) & (col("emp1.name") < col("emp2.name"))
).select(col("emp1.name").alias("Employee_1"), col("emp2.name").alias("Employee_2"), col("emp1.city"))

same_city_df.show()

# ---- OPERATION 5: Window Function — Running Total Salary by Country ----
print("=== Operation 5: Running Total Salary by Country ===")
rt_window = Window.partitionBy("country").orderBy("salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_rt = df.withColumn("running_salary_total", spark_sum("salary").over(rt_window))
df_rt.select("country", "name", "salary", "running_salary_total")\
     .orderBy("country", "running_salary_total")\
     .show()

# ---- OPERATION 6: Filtering with Regex — Names starting with A, B, or C ----
print("=== Operation 6: Regex Filter — Names starting with A, B, or C ===")
df_regex = df.filter(col("name").rlike("^[ABC]"))
df_regex.select("name", "city", "country").show()


=== Operation 1: Window Function — Salary Rank per Country ===
+------+--------+-------+-----------+
|  name| country| salary|salary_rank|
+------+--------+-------+-----------+
| Elena|   Ghana|91000.0|          1|
|Farouk|   Kenya|74000.0|          1|
| Alice|   Kenya|72000.0|          2|
| Grace| Nigeria|88000.0|          1|
|   Bob|Tanzania|85000.0|          1|
| Chloe|Tanzania|68000.0|          2|
|Hassan|Tanzania|65000.0|          3|
| David|  Uganda|79000.0|          1|
+------+--------+-------+-----------+

-> Window functions compute per-group positions WITHOUT collapsing rows — very powerful!

=== Operation 2: UDF — Salary Category ===
+------+-------+-----------+
|  name| salary|salary_tier|
+------+-------+-----------+
| Alice|72000.0|  Mid-level|
|   Bob|85000.0|     Senior|
| Chloe|68000.0|     Junior|
| David|79000.0|  Mid-level|
| Elena|91000.0|     Senior|
|Farouk|74000.0|  Mid-level|
| Grace|88000.0|     Senior|
|Hassan|65000.0|     Junior|
+------+-------+-----------+

+------+-------+-----------+
|  name| salary|salary_tier|
+------+-------+-----------+
| Alice|72000.0|  Mid-level|
|   Bob|85000.0|     Senior|
| Chloe|68000.0|     Junior|
| David|79000.0|  Mid-level|
| Elena|91000.0|     Senior|
|Farouk|74000.0|  Mid-level|
| Grace|88000.0|     Senior|
|Hassan|65000.0|     Junior|
+------+-------+-----------+



+-----------+-------------+----------+
|salary_tier|num_employees|avg_salary|
+-----------+-------------+----------+
|     Junior|            2|   66500.0|
|  Mid-level|            3|   75000.0|
|     Senior|            3|   88000.0|
+-----------+-------------+----------+

-> UDFs let you apply arbitrary Python logic column-by-column in Spark!

=== Operation 3: Pivot Table — Employees per Salary Tier by Country ===


+--------+------+---------+------+
| country|Junior|Mid-level|Senior|
+--------+------+---------+------+
|   Ghana|     0|        0|     1|
|   Kenya|     0|        2|     0|
| Nigeria|     0|        0|     1|
|Tanzania|     2|        0|     1|
|  Uganda|     0|        1|     0|
+--------+------+---------+------+

-> Pivot reshapes long data to wide format — great for cross-tabulations!

=== Operation 4: Self-join — Employees in the Same City ===


+----------+----------+--------+
|Employee_1|Employee_2|    city|
+----------+----------+--------+
|     Chloe|    Hassan|Zanzibar|
|     Alice|    Farouk| Nairobi|
+----------+----------+--------+

=== Operation 5: Running Total Salary by Country ===


+--------+------+-------+--------------------+
| country|  name| salary|running_salary_total|
+--------+------+-------+--------------------+
|   Ghana| Elena|91000.0|             91000.0|
|   Kenya| Alice|72000.0|             72000.0|
|   Kenya|Farouk|74000.0|            146000.0|
| Nigeria| Grace|88000.0|             88000.0|
|Tanzania|Hassan|65000.0|             65000.0|
|Tanzania| Chloe|68000.0|            133000.0|
|Tanzania|   Bob|85000.0|            218000.0|
|  Uganda| David|79000.0|             79000.0|
+--------+------+-------+--------------------+

=== Operation 6: Regex Filter — Names starting with A, B, or C ===


+-----+-------------+--------+
| name|         city| country|
+-----+-------------+--------+
|Alice|      Nairobi|   Kenya|
|  Bob|Dar es Salaam|Tanzania|
|Chloe|     Zanzibar|Tanzania|
+-----+-------------+--------+



In [14]:
# EXERCISE 3: Verify your Delta Lake data in MinIO
# 1. Open http://localhost:9001 in your browser
# 2. Login: admin / bigdata123
# 3. Navigate to Buckets → warehouse
# 4. Take a screenshot showing the delta_log folder
# 5. From Python: verify the Delta table still has correct row count

# Verify all Delta tables we created
for path in ["s3a://warehouse/lab01/employees",
             "s3a://warehouse/lab01/products_parquet"]:
    try:
        count = spark.read.format("delta").load(
            path.replace("parquet", "employees")  # adjust for parquet
        ).count() if "employees" in path else \
               spark.read.parquet(path).count()
        print(f"✅ {path}: {count} rows")
    except Exception as e:
        print(f"❌ {path}: {e}")

✅ s3a://warehouse/lab01/employees: 11 rows
✅ s3a://warehouse/lab01/products_parquet: 6 rows


## Wrap Up

### What You Learned Today:
- ✅ Create a SparkSession connected to a real cluster
- ✅ Create DataFrames from Python data
- ✅ Apply transformations (filter, select, withColumn) and actions (show, count)
- ✅ Aggregate data with groupBy
- ✅ Use Spark SQL
- ✅ Write to Delta Lake format on MinIO (S3)
- ✅ Use Delta Lake time travel and MERGE (UPSERT)
- ✅ Explore the Spark Web UI

### Key Concepts to Remember:
| Concept | Description |
|---------|-------------|
| Transformation | Lazy; returns a new DataFrame; doesn't execute |
| Action | Triggers execution; returns a result |
| Delta Lake | ACID transactions + time travel on object storage |
| MinIO | S3-compatible local object storage (our 'data lake') |
| Spark UI | Your best debugging tool (localhost:8080) |

### Assignment 1 (Due Sunday 15 March, 23:59):
1. Screenshot of `docker-compose ps` showing all services running
2. Complete this notebook (run all cells + add 3 original operations)
3. Load an external CSV dataset, explore it, and write it to Delta Lake
4. Write a 100-word reflection: What surprised you? What questions do you have?
5. Submit: notebook (.ipynb) + screenshot (.png) + reflection (.pdf or .txt)

In [15]:
# Always stop your SparkSession when done (frees cluster resources)
spark.stop()
print("SparkSession stopped. See you next Monday!")

SparkSession stopped. See you next Monday!
